# 02 — Data Preprocessing

**Goal:** turn the three messy Excel files into clean hourly tables that notebook 03 can train
on, and record which values are real rather than filled in.

We are preparing data to forecast **NH4, COD and TSS 4 hours ahead**.

Every step here implements a finding from `01_eda.ipynb`.

**What this notebook saves** into a `processed/` folder:

| File | Contents |
|---|---|
| `{SITE}_hourly.pkl` | hourly values + a matching table of true/false "is this real?" flags |
| `selected_features.json` | which columns to use, per site and per indicator |

## Step 1 — Imports and settings

In [1]:
import datetime as dt
import json
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)

SITE_FILES = {
    "BAY_MAU": "BAY MAU.xlsx",
    "CAU_NGA": "CAU NGA.xlsx",
    "HO_TAY": "HO TAY.xlsx",
}
SITE_NAME = {"BAY_MAU": "Bay Mau", "CAU_NGA": "Cau Nga", "HO_TAY": "Ho Tay"}

# Columns we might use as inputs (the empty ones from notebook 01 are already excluded).
USEFUL_COLUMNS = ["temp", "ph", "tss", "cod", "nh4", "no3", "flow_in", "flow_out1"]

TARGETS = ["nh4", "cod", "tss"]
HORIZON_HOURS = 4

# Fill a hole only if it is this short. Anything longer stays missing.
MAX_HOURS_TO_FILL = 2

# Keep a column as a feature only if its correlation is at least this strong.
MIN_CORRELATION = 0.10

os.makedirs("cache", exist_ok=True)
os.makedirs("processed", exist_ok=True)
print("Settings ready.")

Settings ready.


## Step 2 — Load the raw files (same cache as notebook 01)

In [2]:
def load_excel(site):
    cache_file = "cache/raw_" + site + ".pkl"
    if os.path.exists(cache_file):
        return pd.read_pickle(cache_file)
    table = pd.read_excel(SITE_FILES[site])
    table.to_pickle(cache_file)
    return table


raw_data = {site: load_excel(site) for site in SITE_FILES}
for site in SITE_FILES:
    print(SITE_NAME[site], ":", len(raw_data[site]), "rows")

Bay Mau : 90021 rows
Cau Nga : 77111 rows
Ho Tay : 63322 rows


## Step 3 — Convert every column to numbers

Finding 1 from notebook 01: some cells were turned into dates by Excel. We remove those first,
otherwise they would become enormous numbers.

In [3]:
def to_number(column):
    """Turn a messy column into numbers. Dates and text become missing values."""
    no_dates = column.apply(
        lambda value: np.nan if isinstance(value, (dt.datetime, dt.date)) else value)
    return pd.to_numeric(no_dates, errors="coerce")


def tidy_one_site(site):
    """Sort by time, drop duplicate timestamps, and make every column numeric."""
    table = raw_data[site].copy()
    table["datetime"] = pd.to_datetime(table["datetime"], errors="coerce")
    table = table.dropna(subset=["datetime"])
    table = table.drop_duplicates(subset=["datetime"])
    table = table.sort_values("datetime")

    for column in USEFUL_COLUMNS:
        if column in table.columns:
            table[column] = to_number(table[column])
        else:
            table[column] = np.nan       # column missing at this site

    return table.set_index("datetime")[USEFUL_COLUMNS]


tidy_data = {site: tidy_one_site(site) for site in SITE_FILES}
print("Converted to numbers. Example - Cau Nga:")
tidy_data["CAU_NGA"].head()

Converted to numbers. Example - Cau Nga:


,temp,ph,tss,cod,nh4,no3,flow_in,flow_out1
datetime,,,,,,,,
2024-03-13 08:25:00,22.760000,8.01,9.09,14.00,0.69,11.10,NaN,505.690002
2024-03-13 08:30:00,22.719999,8.01,8.93,14.05,0.69,11.19,NaN,516.599976
2024-03-13 08:35:00,22.690001,8.01,8.74,14.08,0.69,11.31,NaN,123.300003
2024-03-13 08:40:00,22.680000,8.01,8.58,14.14,0.69,11.31,NaN,28.860001
2024-03-13 08:45:00,22.680000,8.01,8.60,14.14,0.69,11.31,NaN,2.570000


## Step 4 — Remove impossible readings

Finding 4 from notebook 01. We remove only values that cannot physically exist. We deliberately
**keep** `flow = 0`, because that simply means the pump is off.

In [4]:
def remove_impossible_values(table):
    """Set physically impossible readings to missing. Flow zeros are left alone."""
    table = table.copy()

    # pH must be between 0 and 14, and a real reading is never exactly 0.
    table.loc[table["ph"] <= 0, "ph"] = np.nan
    table.loc[table["ph"] > 14, "ph"] = np.nan

    # Water temperature in Hanoi is never 0 or above 50 degrees.
    table.loc[table["temp"] <= 0, "temp"] = np.nan
    table.loc[table["temp"] > 50, "temp"] = np.nan

    # Concentrations and flows cannot be negative.
    for column in ["tss", "cod", "nh4", "no3", "flow_in", "flow_out1"]:
        table.loc[table[column] < 0, column] = np.nan

    return table


for site in SITE_FILES:
    before = tidy_data[site].notna().sum().sum()
    tidy_data[site] = remove_impossible_values(tidy_data[site])
    after = tidy_data[site].notna().sum().sum()
    print(SITE_NAME[site], ": removed", before - after, "impossible readings")

Bay Mau : removed 2 impossible readings
Cau Nga : removed 8 impossible readings
Ho Tay : removed 10 impossible readings


## Step 5 — Average into hours, and remember what is real

This is the key step for honest results.

We build **two** tables that line up row for row:

- `values` — the average of the readings in each hour
- `is_real` — `True` if that hour contained at least one genuine sensor reading

Finding 3 from notebook 01 is handled here. We fill holes of up to 2 hours so that the lag
features in notebook 03 can still be calculated, **but the filled hours stay marked `False`**,
so notebook 04 will never measure accuracy on them.

In [5]:
def make_hourly(table):
    """Return hourly averages plus a table saying which hours are genuine."""
    values = table.resample("1h").mean()

    # An hour is real if it contained at least one actual reading.
    is_real = table.resample("1h").count() > 0

    # Fill only short holes, so lag features remain computable.
    values = values.interpolate(method="time",
                                limit=MAX_HOURS_TO_FILL,
                                limit_area="inside")

    return values, is_real


hourly_values = {}
hourly_is_real = {}

for site in SITE_FILES:
    values, is_real = make_hourly(tidy_data[site])
    hourly_values[site] = values
    hourly_is_real[site] = is_real
    print(SITE_NAME[site], ":", len(values), "hours from",
          values.index.min().date(), "to", values.index.max().date())

Bay Mau : 8784 hours from 2024-01-01 to 2024-12-31
Cau Nga : 7048 hours from 2024-03-13 to 2024-12-31
Ho Tay : 7048 hours from 2024-03-13 to 2024-12-31


In [6]:
# How much of each indicator is genuinely measured?
report = []
for site in SITE_FILES:
    row = {"site": SITE_NAME[site], "total_hours": len(hourly_values[site])}
    for target in TARGETS:
        row["real_" + target] = round(hourly_is_real[site][target].mean(), 3)
    report.append(row)

print("Fraction of hours that contain a real reading:")
pd.DataFrame(report).set_index("site")

Fraction of hours that contain a real reading:


,total_hours,real_nh4,real_cod,real_tss
site,,,,
Bay Mau,8784,0.627,0.837,0.830
Cau Nga,7048,0.915,0.915,0.915
Ho Tay,7048,0.699,0.758,0.707


Ho Tay is around 0.70 and drops much lower in the most recent months, which is exactly the
period notebook 03 will use for testing. Keeping the `is_real` flag is what stops us from
reporting a fake accuracy score there.

## Step 6 — Choose the input columns for each indicator

Finding 6 from notebook 01, turned into a rule:

> Keep a column if it is present at this site **and** its correlation with the indicator
> 4 hours later is at least 0.10 in size. Always keep the indicator's own history.

Doing this per site matters because `no3` and `flow_in` do not exist everywhere.

In [7]:
def choose_features(site, target):
    """Pick input columns for one site and one indicator, based on measured correlation."""
    values = hourly_values[site]
    future_target = values[target].shift(-HORIZON_HOURS)

    chosen = []
    for column in USEFUL_COLUMNS:
        # Skip columns that barely exist at this site.
        if values[column].notna().mean() < 0.5:
            continue

        correlation = values[column].corr(future_target)
        if pd.notna(correlation) and abs(correlation) >= MIN_CORRELATION:
            chosen.append(column)

    # The indicator's own past is always allowed.
    if target not in chosen:
        chosen = [target] + chosen

    return chosen


selected_features = {}
for site in SITE_FILES:
    selected_features[site] = {}
    print(SITE_NAME[site])
    for target in TARGETS:
        selected_features[site][target] = choose_features(site, target)
        dropped = [c for c in USEFUL_COLUMNS if c not in selected_features[site][target]]
        print("   ", target, "-> keep:", selected_features[site][target])
        print("       drop:", dropped)

Bay Mau
    nh4 -> keep: ['temp', 'tss', 'cod', 'nh4']
       drop: ['ph', 'no3', 'flow_in', 'flow_out1']
    cod -> keep: ['ph', 'tss', 'cod', 'nh4']
       drop: ['temp', 'no3', 'flow_in', 'flow_out1']
    tss -> keep: ['temp', 'tss', 'cod', 'nh4']
       drop: ['ph', 'no3', 'flow_in', 'flow_out1']
Cau Nga
    nh4 -> keep: ['ph', 'cod', 'nh4']
       drop: ['temp', 'tss', 'no3', 'flow_in', 'flow_out1']
    cod -> keep: ['temp', 'tss', 'cod', 'nh4', 'no3']
       drop: ['ph', 'flow_in', 'flow_out1']
    tss -> keep: ['temp', 'ph', 'tss', 'cod']
       drop: ['nh4', 'no3', 'flow_in', 'flow_out1']
Ho Tay
    nh4 -> keep: ['ph', 'tss', 'cod', 'nh4', 'no3']
       drop: ['temp', 'flow_in', 'flow_out1']
    cod -> keep: ['ph', 'tss', 'cod', 'nh4', 'no3']
       drop: ['temp', 'flow_in', 'flow_out1']
    tss -> keep: ['temp', 'ph', 'tss', 'cod', 'nh4', 'no3']
       drop: ['flow_in', 'flow_out1']


`flow_in` and `flow_out1` are dropped for every site and every indicator. The original LSTM
notebook used `flow_out1` as an input everywhere; the data says it does not belong there.

## Step 7 — Save everything for the next notebook

In [8]:
for site in SITE_FILES:
    bundle = {
        "values": hourly_values[site],
        "is_real": hourly_is_real[site],
    }
    pd.to_pickle(bundle, "processed/" + site + "_hourly.pkl")
    print("saved processed/" + site + "_hourly.pkl")

with open("processed/selected_features.json", "w") as file:
    json.dump(selected_features, file, indent=2)
print("saved processed/selected_features.json")

saved processed/BAY_MAU_hourly.pkl
saved processed/CAU_NGA_hourly.pkl
saved processed/HO_TAY_hourly.pkl
saved processed/selected_features.json


## Summary

| Step | What we did | Why |
|---|---|---|
| 3 | converted columns to numbers | Excel had stray date cells |
| 4 | removed impossible values | pH 0 and 0 degrees are sensor faults; flow 0 is normal |
| 5 | hourly averages + `is_real` flag | less noise, and honest accuracy later |
| 6 | chose features by correlation | stop guessing which columns matter |

Next: **`03_model_training.ipynb`**.